# AI Code Auditor — QLoRA Fine-tuning

**Model:** CodeLlama-7b-hf  
**Method:** QLoRA (4-bit NF4 + LoRA r=16)  
**Dataset:** Big-Vul v2 (improved prompts, token-filtered)  
**Expected runtime:** ~4-5 hours on Kaggle P100

### Before running:
1. Set GPU to **P100** in Notebook settings (right panel)
2. Enable **Internet** in Notebook settings
3. Add `HF_TOKEN` in Add-ons → Secrets
4. Upload `train.jsonl` and `val.jsonl` as a Kaggle Dataset

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
# Pin exact versions known to work on Kaggle
!pip install -q \
    transformers==4.40.2 \
    peft==0.10.0 \
    trl==0.8.6 \
    bitsandbytes==0.43.1 \
    accelerate==0.29.3 \
    datasets==2.19.1 \
    loguru
print('Done')

In [ ]:
# ── Cell 3: HuggingFace login ──────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

try:
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    login(token=hf_token)
    print('Logged in to HuggingFace')
except Exception as e:
    print(f'No HF secret found ({e}). Proceeding without auth.')

In [ ]:
# ── Cell 4: Dataset paths ──────────────────────────────────────────────────
import os, json

TRAIN_PATH = None
VAL_PATH   = None

for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        full = os.path.join(root, f)
        if f == 'train.jsonl': TRAIN_PATH = full
        if f == 'val.jsonl':   VAL_PATH   = full

assert TRAIN_PATH and VAL_PATH, 'Could not find train.jsonl / val.jsonl'
print(f'Train: {TRAIN_PATH}')
print(f'Val:   {VAL_PATH}')

with open(TRAIN_PATH) as f:
    sample = json.loads(f.readline())
print(f'Sample keys: {list(sample.keys())}')
print(f'CWE: {sample["cwe"]} | Text length: {len(sample["text"])} chars')
print(f'\nCompletion preview (first 300 chars):')
print(sample['completion'][:300])

In [ ]:
# ── Cell 5: Config ─────────────────────────────────────────────────────────
CONFIG = {
    # Model — 7B with QLoRA fits in ~6-7GB VRAM on P100
    'base_model': 'codellama/CodeLlama-7b-hf',
    'output_dir': '/kaggle/working/lora_adapter',

    # QLoRA — 4-bit NF4 quantization
    'load_in_4bit': True,
    'bnb_4bit_quant_type': 'nf4',
    'bnb_4bit_compute_dtype': 'float16',
    'bnb_4bit_use_double_quant': True,

    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],

    # Training — tuned for P100 16GB with 7B 4-bit model
    'num_epochs': 3,
    'batch_size': 4,
    'grad_accum': 4,          # effective batch = 16
    'learning_rate': 2e-4,
    'max_seq_length': 768,    # fits token-filtered dataset cleanly
    'warmup_ratio': 0.03,
    'lr_scheduler': 'cosine',
    'logging_steps': 25,
    'eval_steps': 100,
    'save_steps': 200,
    'seed': 42,
}
print('Config ready')
print(f'Model: {CONFIG["base_model"]}')

In [ ]:
# ── Cell 6: Load model + tokenizer ─────────────────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print(f'CUDA: {torch.cuda.is_available()} | Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

tokenizer = AutoTokenizer.from_pretrained(
    CONFIG['base_model'], trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit NF4...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG['base_model'],
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.use_cache = False
model.config.pretraining_tp = 1

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved() / 1e9
print(f'GPU memory — allocated: {allocated:.1f}GB | reserved: {reserved:.1f}GB')

In [ ]:
# ── Cell 7: Apply LoRA ─────────────────────────────────────────────────────
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

# prepare_model_for_kbit_training works correctly with 4-bit quantized models
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=CONFIG['target_modules'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5% trainable — e.g. trainable params: 39,976,960 || all params: 6,778,671,104

In [ ]:
# ── Cell 8: Load datasets ──────────────────────────────────────────────────
from datasets import load_dataset

train_dataset = load_dataset('json', data_files=TRAIN_PATH, split='train')
val_dataset   = load_dataset('json', data_files=VAL_PATH,   split='train')

print(f'Train: {len(train_dataset):,} samples')
print(f'Val:   {len(val_dataset):,} samples')
print(f'\nSample text (first 400 chars):')
print(train_dataset[0]['text'][:400])

In [ ]:
# ── Cell 9: Train ──────────────────────────────────────────────────────────
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['batch_size'],
    per_device_eval_batch_size=CONFIG['batch_size'],
    gradient_accumulation_steps=CONFIG['grad_accum'],
    gradient_checkpointing=True,
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    warmup_ratio=CONFIG['warmup_ratio'],
    fp16=True,
    logging_steps=CONFIG['logging_steps'],
    evaluation_strategy='steps',
    eval_steps=CONFIG['eval_steps'],
    save_strategy='steps',
    save_steps=CONFIG['save_steps'],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    optim='paged_adamw_32bit',
    group_by_length=True,
    report_to='none',
    seed=CONFIG['seed'],
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    args=training_args,
    dataset_text_field='text',
    max_seq_length=CONFIG['max_seq_length'],
    packing=False,
)

print('Starting training...')
print(f'Train samples : {len(train_dataset):,}')
print(f'Steps per epoch: {len(train_dataset) // (CONFIG["batch_size"] * CONFIG["grad_accum"])}')
print(f'Total steps   : {len(train_dataset) // (CONFIG["batch_size"] * CONFIG["grad_accum"]) * CONFIG["num_epochs"]}')
trainer.train()

In [ ]:
# ── Cell 10: Save adapter + tokenizer ─────────────────────────────────────
from pathlib import Path

output_dir = Path(CONFIG['output_dir'])
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f'LoRA adapter saved to {output_dir}')
print('Files:', [f.name for f in output_dir.iterdir()])

In [ ]:
# ── Cell 10b: Auto-backup to Google Drive ─────────────────────────────────
# This runs immediately after saving — protects against session expiry
import shutil, os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_BACKUP = '/content/drive/MyDrive/ai_code_auditor'
except ImportError:
    # On Kaggle — use /kaggle/working (auto-persisted on Save Version)
    DRIVE_BACKUP = None
    print('Not on Colab — skipping Drive mount')
    print('On Kaggle: click Save Version (top right) to persist outputs')

if DRIVE_BACKUP:
    os.makedirs(DRIVE_BACKUP, exist_ok=True)
    # Zip and copy adapter
    zip_path = f'{DRIVE_BACKUP}/lora_adapter_codellama7b'
    shutil.make_archive(zip_path, 'zip', str(output_dir))
    print(f'Adapter backed up to Google Drive: {zip_path}.zip')

# Also zip locally for Kaggle Output tab download
local_zip = '/kaggle/working/lora_adapter_download'
shutil.make_archive(local_zip, 'zip', str(output_dir))
zip_size = Path(local_zip + '.zip').stat().st_size / 1e6
print(f'Local zip ready: lora_adapter_download.zip ({zip_size:.0f} MB)')
print('Download it from the Output tab NOW as backup!')

In [ ]:
# ── Cell 11: Save training log + plot ─────────────────────────────────────
import json, matplotlib.pyplot as plt

log_history = trainer.state.log_history

# Save log
with open('/kaggle/working/training_log.json', 'w') as f:
    json.dump(log_history, f, indent=2)

# Plot
train_loss = [(e['step'], e['loss'])      for e in log_history if 'loss'      in e and 'eval_loss' not in e]
eval_loss  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(10, 4))
if train_loss:
    steps, losses = zip(*train_loss)
    ax.plot(steps, losses, label='Train Loss', color='steelblue', alpha=0.7)
if eval_loss:
    steps, losses = zip(*eval_loss)
    ax.plot(steps, losses, label='Eval Loss', color='coral', linestyle='--', marker='o')
ax.set_title('QLoRA Training — CodeLlama-7B on Big-Vul', fontweight='bold')
ax.set_xlabel('Step'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/training_loss.png', dpi=150)
plt.show()
print('Saved training_loss.png')

In [ ]:
# ── Cell 12: Quick inference test ─────────────────────────────────────────
import torch

merged_model = trainer.model
merged_model.eval()

TEST_CODE = """void process_input(char *user_input) {
    char buffer[128];
    strcpy(buffer, user_input);
    printf(\"Processing: %s\\n\", buffer);
}"""

prompt = (
    "<s>[INST] <<SYS>>\n"
    "You are an expert security code auditor. Identify vulnerabilities and rewrite securely.\n"
    "<</SYS>>\n\n"
    f"Analyze the following C/C++ code for security vulnerabilities and provide a secure rewrite:\n\n"
    f"```c\n{TEST_CODE}\n``` [/INST]"
)

inputs = tokenizer(prompt, return_tensors='pt').to(merged_model.device)

with torch.no_grad():
    outputs = merged_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

generated = outputs[0][inputs['input_ids'].shape[1]:]
print(tokenizer.decode(generated, skip_special_tokens=True))

In [ ]:
# ── Cell 13: Zip adapter for download ─────────────────────────────────────
import shutil, json
from pathlib import Path

# Zip the lora_adapter folder so it can be downloaded as one file
shutil.make_archive('/kaggle/working/lora_adapter_download', 'zip', '/kaggle/working/lora_adapter')

print('All outputs in /kaggle/working/')
print()
print('Files saved:')
for p in Path('/kaggle/working').iterdir():
    if p.is_file():
        print(f'  {p.name} ({p.stat().st_size / 1e6:.1f} MB)')
    elif p.is_dir() and p.name != '.virtual_documents':
        size = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())
        print(f'  {p.name}/ ({size / 1e6:.1f} MB)')

print()
print('Download from Output tab:')
print('  lora_adapter_download.zip  <- main model')
print('  training_log.json')
print('  training_loss.png')